# **Applicazione massiva di BERT**

> Si tratta di un notebook ausiliario, a cui si ricorre durante l'esecuzione di `3_nlp_sentiment__recensioni.ipynb` per l'applicazione massiva di BERT sull'ambiente Google Colab, così da sfruttare la GPU che mette a disposizione.

In [2]:
!pip install transformers torch pandas tqdm sentencepiece -q

import torch
import pandas as pd
from google.colab import files
from transformers import pipeline, AutoTokenizer

In [3]:
# Verifica GPU: con max_length=512 l'inferenza su CPU sarebbe proibitiva
device = 0 if torch.cuda.is_available() else -1
if device == -1:
    raise RuntimeError("GPU non attiva. Vai su Runtime → Cambia tipo di runtime → GPU (T4).")

print(f"Dispositivo: GPU attiva — {torch.cuda.get_device_name(0)}.")

# Caricamento del dataset esportato dal notebook locale
df = pd.read_csv("3_1_input_per_colab.csv", index_col="ID Recensione")
print(f"Dataset caricato: {len(df):,} righe.")

# Il file esportato contiene già solo righe valutabili, ma si ricontrolla per sicurezza
df_valido = df[
    df["Corpus_Sentiment"].notna() &
    (df["Corpus_Sentiment"].astype(str).str.strip() != "")].copy()

testi = df_valido["Corpus_Sentiment"].astype(str).tolist()
print(f"Righe da analizzare: {len(testi):,}.")

Dispositivo: GPU attiva — Tesla T4.
Dataset caricato: 97,240 righe.
Righe da analizzare: 97,240.


In [ ]:
import time
import numpy as np

MODELLO = "neuraly/bert-base-italian-cased-sentiment"
BATCH   = 64

pipe = pipeline("sentiment-analysis", model=MODELLO, device=0, top_k=None, truncation=True, max_length=512)

print(f"Testi da elaborare: {len(testi):,}", flush=True)

# Ordinamento per lunghezza: batch omogenei, molto meno padding sprecato. L'ordine originale viene ripristinato a fine inferenza tramite la lista 'ordine'.
ordine = sorted(range(len(testi)), key=lambda i: len(testi[i]))
testi_ordinati = [testi[i] for i in ordine]
print("Ordinamento completato.", flush=True)

# Stima su 5 batch: quanto ci vorrà davvero?
t0 = time.time()
_ = pipe(testi_ordinati[:BATCH * 5])
t_prova = time.time() - t0
n_batch_tot = int(np.ceil(len(testi) / BATCH))
stima_min = (t_prova / 5) * n_batch_tot / 60
print(f"Tempo per 5 batch: {t_prova:.1f}s → stima totale: {stima_min:.1f} minuti ({n_batch_tot:,} batch)\n", flush=True)

# Inferenza con progresso stampato ogni 100 batch (niente tqdm: stampa forzata)
pos_ord, neu_ord, neg_ord = [], [], []
t_start = time.time()

for k, i in enumerate(range(0, len(testi_ordinati), BATCH)):
    out_batch = pipe(testi_ordinati[i:i + BATCH])
    for out in out_batch:
        prob = {r["label"].lower(): r["score"] for r in out}
        pos_ord.append(prob.get("positive", 0.0))
        neu_ord.append(prob.get("neutral",  0.0))
        neg_ord.append(prob.get("negative", 0.0))

    if k % 100 == 0:
        trascorso = time.time() - t_start
        fatti = len(pos_ord)
        pct = fatti / len(testi) * 100
        eta = (trascorso / max(fatti, 1)) * (len(testi) - fatti) / 60
        print(f"  batch {k:,}/{n_batch_tot:,} | {fatti:,} testi ({pct:.1f}%) | "
              f"trascorsi {trascorso/60:.1f} min | ETA {eta:.1f} min", flush=True)

print(f"\nCompletato in {(time.time()-t_start)/60:.1f} minuti.", flush=True)

# Ripristino ordine originale
n = len(testi)
p_pos, p_neu, p_neg = [0.0]*n, [0.0]*n, [0.0]*n
for posizione, idx_orig in enumerate(ordine):
    p_pos[idx_orig] = pos_ord[posizione]
    p_neu[idx_orig] = neu_ord[posizione]
    p_neg[idx_orig] = neg_ord[posizione]

df_valido["P_pos_BERT"] = p_pos
df_valido["P_neu_BERT"] = p_neu
df_valido["P_neg_BERT"] = p_neg

# Score atteso (soft score): E[S] = (+1)*P(pos) + 0*P(neu) + (-1)*P(neg) = P(pos) - P(neg)
df_valido["Sentiment_BERT_Soft"] = (df_valido["P_pos_BERT"] - df_valido["P_neg_BERT"]).round(4)
print("Score calcolati.")

config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

...

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Testi da elaborare: 97,240
Ordinamento completato.
Tempo per 5 batch: 3.8s → stima totale: 19.0 minuti (1,520 batch)

  batch 0/1,520 | 64 testi (0.1%) | trascorsi 0.0 min | ETA 19.0 min


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  batch 100/1,520 | 6,464 testi (6.6%) | trascorsi 1.1 min | ETA 14.9 min
  batch 200/1,520 | 12,864 testi (13.2%) | trascorsi 2.1 min | ETA 13.6 min
  batch 300/1,520 | 19,264 testi (19.8%) | trascorsi 3.3 min | ETA 13.5 min
  batch 400/1,520 | 25,664 testi (26.4%) | trascorsi 4.6 min | ETA 12.7 min
  batch 500/1,520 | 32,064 testi (33.0%) | trascorsi 5.8 min | ETA 11.8 min
  batch 600/1,520 | 38,464 testi (39.6%) | trascorsi 6.8 min | ETA 10.4 min
  batch 700/1,520 | 44,864 testi (46.1%) | trascorsi 7.8 min | ETA 9.1 min
  batch 800/1,520 | 51,264 testi (52.7%) | trascorsi 8.8 min | ETA 7.9 min
  batch 900/1,520 | 57,664 testi (59.3%) | trascorsi 9.9 min | ETA 6.8 min
  batch 1,000/1,520 | 64,064 testi (65.9%) | trascorsi 10.9 min | ETA 5.7 min
  batch 1,100/1,520 | 70,464 testi (72.5%) | trascorsi 11.9 min | ETA 4.5 min
  batch 1,200/1,520 | 76,864 testi (79.0%) | trascorsi 13.0 min | ETA 3.4 min
  batch 1,300/1,520 | 83,264 testi (85.6%) | trascorsi 14.0 min | ETA 2.4 min
  batch 1

In [ ]:
# Si salvano anche le probabilità grezze, per replicare sul dataset completo la diagnostica sulla neutralità condotta sul sottocampione.
NOME_FILE = "3_2_bert_soft.csv"
df_valido[["Sentiment_BERT_Soft", "P_pos_BERT", "P_neu_BERT", "P_neg_BERT"]].to_csv(NOME_FILE)

s = df_valido["Sentiment_BERT_Soft"]
print(f"Salvato: {NOME_FILE} ({len(df_valido):,} righe).\n")
print(f"Media: {s.mean():.3f} | Mediana: {s.median():.3f}")
print(f"Positivi (> 0.05):  {(s >  0.05).sum():,}")
print(f"Neutri:             {((s >= -0.05) & (s <= 0.05)).sum():,}")
print(f"Negativi (< -0.05): {(s < -0.05).sum():,}")

# Diagnostica sulla neutralità: uno score ≈ 0 indica neutralità netta o incertezza?
quasi_zero    = df_valido[df_valido["Sentiment_BERT_Soft"].abs() < 0.1]
neutri_netti  = (quasi_zero["P_neu_BERT"] > 0.5).sum()
incerti_polar = (quasi_zero["P_neu_BERT"] <= 0.5).sum()

print(f"\nDiagnostica sugli score prossimi a zero (|score| < 0.1): {len(quasi_zero):,} recensioni")
print(f"  - neutralità netta   (P_neu > 0.5): {neutri_netti:,} ({neutri_netti/max(len(quasi_zero),1)*100:.1f}%)")
print(f"  - incertezza polarizzata:           {incerti_polar:,} ({incerti_polar/max(len(quasi_zero),1)*100:.1f}%)")

# Diagnostica sul troncamento — sull'intero corpus
tok = AutoTokenizer.from_pretrained(MODELLO)
lunghezze_tot = [len(tok.encode(t, truncation=False)) for t in testi]
oltre_512_tot = sum(1 for l in lunghezze_tot if l > 512)
print(f"\nDiagnostica troncamento (intero corpus, {len(testi):,} recensioni):")
print(f"  - lunghezza media:  {sum(lunghezze_tot)/len(lunghezze_tot):.1f} token")
print(f"  - lunghezza massima: {max(lunghezze_tot):,} token")
print(f"  - oltre 512 token: {oltre_512_tot:,} ({oltre_512_tot/len(testi)*100:.2f}%)")

Salvato: 3_2_bert_soft.csv (97,240 righe).

Media: 0.776 | Mediana: 0.998
Positivi (> 0.05):  86,211
Neutri:             2,214
Negativi (< -0.05): 8,815

Diagnostica sugli score prossimi a zero (|score| < 0.1): 3,029 recensioni
  - neutralità netta   (P_neu > 0.5): 2,789 (92.1%)
  - incertezza polarizzata:           240 (7.9%)

Diagnostica troncamento (intero corpus, 97,240 recensioni):
  - lunghezza media:  19.8 token
  - lunghezza massima: 366 token
  - oltre 512 token: 0 (0.00%)
